In [ ]:
# Setup for Kaggle or Google Colab environments
# Run this cell to install any missing dependencies!
!pip install -q pandas numpy scikit-learn scipy matplotlib seaborn mplsoccer tqdm

# Task 1 — Extract Shot Data from StatsBomb (FIFA World Cup 2018 + 2022)

**Objective:** the original dataset is too large (many nested JSON files), so this step only extracts the necessary fields for the **Shot Zone / Shot Quality Clustering** (K-Means → KNN) from all matches of **FIFA World Cup 2018 and 2022**, and writes them to a flat `.csv` file so that subsequent tasks (descriptive statistics, EDA, preprocessing, modeling) can reuse it without touching the original JSONs.

**Data scope:** fixed **FIFA World Cup**, `competition_id = 43`, getting **both full 64-match seasons**:
- `season_id = 3` → **World Cup 2018**
- `season_id = 106` → **World Cup 2022**

(These are the only 2 World Cup seasons in the StatsBomb open-data with a full **64 matches**; older World Cups from 1958-1990 only have a few matches so they are not used here.)

**Data source:** Kaggle Dataset **`saurabhshahane/statsbomb-football-data`** — when you add this dataset to the notebook, Kaggle will mount it at `/kaggle/input/statsbomb-football-data/`.

**Important Note:** I do not have direct access to Kaggle to view the exact folder structure inside that dataset, so this notebook has an **auto-discovery** step — it automatically scans the entire input directory to find `competitions.json`, the `matches/{competition_id}/{season_id}.json` files and `events/{match_id}.json`, regardless of how deep they are nested. This ensures the notebook runs correctly without manual path adjustments. If the dataset structure is too different, the auto-discovery cell will print a specific message so you know what needs fixing.


## 1.0 — Import libraries

In [ ]:
import json
import math
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm


## 1.0 (cont) — Config

In [ ]:
# ---------------- Config ----------------

# Fix FIFA World Cup
COMPETITION_ID = 43

# World Cup 2018 (season_id=3) and World Cup 2022 (season_id=106) - both have full 64 matches
SEASON_IDS = [3, 106]

# Limit the number of matches to process PER season (for quick testing). Set to None to run all 64+64=128 matches.
MAX_MATCHES_PER_SEASON = None   # change to e.g., 5 if you want to test quickly first

# Kaggle Dataset input path (the exact slug you added: saurabhshahane/statsbomb-football-data)
KAGGLE_INPUT_DIR = Path("/kaggle/input/datasets/saurabhshahane/statsbomb-football-data/data")

# Output directory on Kaggle
WORK_DIR = Path("/kaggle/working")
OUTPUT_CSV = WORK_DIR / "shots_worldcup_2018_2022_raw.csv"

# Goal coordinates (StatsBomb standard pitch coordinates 120 x 80)
GOAL_X, GOAL_Y = 120.0, 40.0


## 1.1 — Auto-discovery: automatically detect dataset folder structure

Scan `KAGGLE_INPUT_DIR` once, index the true locations of:
- `competitions.json`
- each file `matches/{competition_id}/{season_id}.json`
- each file `events/{match_id}.json`

This approach does not depend on whether the dataset is wrapped in a `data/` folder or has a dataset name outside.

In [ ]:
def discover_paths(base_dir: Path):
    competitions_path = None
    matches_index = {}   # (competition_id, season_id) -> Path
    events_index = {}    # match_id (str) -> Path

    all_json_files = list(base_dir.rglob("*.json"))

    for p in all_json_files:
        parts = p.parts

        if p.name == "competitions.json" and competitions_path is None:
            competitions_path = p
            continue

        # matches/{competition_id}/{season_id}.json
        if len(parts) >= 3 and parts[-3].lower() == "matches":
            competition_id, season_id = parts[-2], p.stem
            if competition_id.isdigit() and season_id.isdigit():
                matches_index[(int(competition_id), int(season_id))] = p
                continue

        # events/{match_id}.json
        if len(parts) >= 2 and parts[-2].lower() == "events":
            match_id = p.stem
            if match_id.isdigit():
                events_index[match_id] = p

    return competitions_path, matches_index, events_index, len(all_json_files)


assert KAGGLE_INPUT_DIR.exists(), (
    f"Directory not found {KAGGLE_INPUT_DIR}. "
    "Please verify you have added the correct dataset 'saurabhshahane/statsbomb-football-data' "
    "to the notebook (the '+ Add Input' button on the right), and the correct dataset slug."
)

competitions_path, matches_index, events_index, n_json_files = discover_paths(KAGGLE_INPUT_DIR)

print(f"Total .json files scanned in dataset: {n_json_files}")
print(f"competitions.json  : {competitions_path}")
print(f"Number of matches entries indexed : {len(matches_index)}")
print(f"Number of events entries indexed  : {len(events_index)}")

assert competitions_path is not None, "competitions.json not found in dataset — verify folder structure."


## 1.2 — Function to load data via the discovered index

In [ ]:
def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def get_match_ids(competition_id: int, season_id: int):
    matches_path = matches_index.get((competition_id, season_id))
    if matches_path is None:
        raise FileNotFoundError(
            f"Matches file not found for competition_id={competition_id}, season_id={season_id}"
        )
    matches = load_json(matches_path)
    return [m["match_id"] for m in matches]


def get_events(match_id: int):
    events_path = events_index.get(str(match_id))
    if events_path is None:
        raise FileNotFoundError(f"Events file not found for match_id={match_id}")
    return load_json(events_path)


## 1.3 — Extract necessary fields from each shot event + calculate derived features

Fields are divided into 3 groups (per agreed design):

| Group | Field |
|---|---|
| **Identifier** | `event_id`, `match_id`, `season_id`, `team_id/name`, `player_id/name`, `period`, `minute`, `second`, `play_pattern` |
| **Features for model (X)** | `location_x/y`, `distance_to_goal`, `angle_to_goal`, `under_pressure`, `body_part`, `technique`, `shot_type`, `first_time`, `aerial_won`, `open_goal`, `n_teammates_in_frame`, `n_opponents_in_frame`, `keeper_x/y`, `end_location_x/y` |
| **Hidden fields to validate clusters later (NOT included in model)** | `outcome`, `statsbomb_xg` |

`distance_to_goal` and `angle_to_goal` are pre-calculated from `location` and goal coordinates `(120, 40)`.

In [ ]:
def compute_distance_angle(x, y):
    dx = GOAL_X - x
    dy = GOAL_Y - y
    distance = math.hypot(dx, dy)
    angle = math.atan2(dy, dx)
    return distance, angle


def extract_shots_from_match(match_id: int, season_id: int):
    events = get_events(match_id)
    rows = []

    for e in events:
        if e.get("type", {}).get("name") != "Shot":
            continue

        shot = e.get("shot", {})
        loc = e.get("location", [None, None])
        x, y = loc[0], loc[1]

        distance_to_goal, angle_to_goal = (None, None)
        if x is not None and y is not None:
            distance_to_goal, angle_to_goal = compute_distance_angle(x, y)

        freeze_frame = shot.get("freeze_frame", []) or []
        n_teammates_in_frame = sum(1 for p in freeze_frame if p.get("teammate") is True)
        n_opponents_in_frame = sum(1 for p in freeze_frame if p.get("teammate") is False)

        keeper_entries = [
            p for p in freeze_frame
            if p.get("teammate") is False and p.get("position", {}).get("name") == "Goalkeeper"
        ]
        keeper_x = keeper_entries[0]["location"][0] if keeper_entries else None
        keeper_y = keeper_entries[0]["location"][1] if keeper_entries else None

        end_loc = shot.get("end_location", [None, None, None]) or [None, None, None]

        row = {
            # --- Identifier ---
            "event_id": e.get("id"),
            "match_id": match_id,
            "season_id": season_id,
            "team_id": e.get("team", {}).get("id"),
            "team_name": e.get("team", {}).get("name"),
            "player_id": e.get("player", {}).get("id"),
            "player_name": e.get("player", {}).get("name"),
            "period": e.get("period"),
            "minute": e.get("minute"),
            "second": e.get("second"),
            "play_pattern": e.get("play_pattern", {}).get("name"),

            # --- Features for model (X) ---
            "location_x": x,
            "location_y": y,
            "distance_to_goal": distance_to_goal,
            "angle_to_goal": angle_to_goal,
            "under_pressure": bool(e.get("under_pressure", False)),
            "body_part": shot.get("body_part", {}).get("name"),
            "technique": shot.get("technique", {}).get("name"),
            "shot_type": shot.get("type", {}).get("name"),
            "first_time": bool(shot.get("first_time", False)),
            "aerial_won": bool(shot.get("aerial_won", False)),
            "open_goal": bool(shot.get("open_goal", False)),
            "n_teammates_in_frame": n_teammates_in_frame,
            "n_opponents_in_frame": n_opponents_in_frame,
            "keeper_x": keeper_x,
            "keeper_y": keeper_y,
            "end_location_x": end_loc[0],
            "end_location_y": end_loc[1],

            # --- Hidden fields to validate clusters later (NOT used as features) ---
            "outcome": shot.get("outcome", {}).get("name"),
            "statsbomb_xg": shot.get("statsbomb_xg"),
        }
        rows.append(row)

    return rows


## 1.4 — Run extraction for all matches of World Cup 2018 + 2022

In [ ]:
all_rows = []
failed_matches = []

for season_id in SEASON_IDS:
    match_ids = get_match_ids(COMPETITION_ID, season_id)
    if MAX_MATCHES_PER_SEASON is not None:
        match_ids = match_ids[:MAX_MATCHES_PER_SEASON]

    print(f"Season {season_id}: {len(match_ids)} matches will be processed")

    for match_id in tqdm(match_ids, desc=f"Season {season_id}"):
        try:
            rows = extract_shots_from_match(match_id, season_id)
            all_rows.extend(rows)
        except Exception as ex:
            print(f"[WARN] Error at match_id={match_id}: {ex}")
            failed_matches.append(match_id)

print(f"\nTotal shots extracted: {len(all_rows)}")
if failed_matches:
    print(f"Number of failed matches (skipped): {len(failed_matches)} -> {failed_matches}")


In [ ]:
df = pd.DataFrame(all_rows)
df.shape


## 1.5 — Quick check before saving (sanity check)

- Are the number of rows / columns reasonable (expected ~1300-1400 shots for 128 matches, based on the average ~0.79% event being a Shot)?
- Are the number of shots per season (2018 vs 2022) balanced (both are 64 matches, so the number of shots should be similar)?
- Data types of each column, missing ratios.
- Does the encoding of accented player names read correctly?

In [ ]:
print("Shape:", df.shape)
df.dtypes


In [ ]:
df["season_id"].value_counts()


In [ ]:
missing = df.isna().sum()
missing[missing > 0].sort_values(ascending=False)


In [ ]:
print(df["outcome"].value_counts())
print("\nstatsbomb_xg describe:")
print(df["statsbomb_xg"].describe())


In [ ]:
sample_accented = df[df["player_name"].str.contains("í|é|ó|ñ|ü|ã", regex=True, na=False)]["player_name"].unique()
sample_accented[:10]


## 1.6 — Write out to CSV file

Use `utf-8-sig` encoding so Excel/pandas can read accented player names without font errors.

In [ ]:
WORK_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved: {OUTPUT_CSV}  ({df.shape[0]} rows, {df.shape[1]} columns)")


In [ ]:
check_df = pd.read_csv(OUTPUT_CSV, encoding="utf-8-sig")
check_df.head(5)


## If auto-discovery reports an error / cannot find the dataset

1. Run the following command in a separate cell to see the true dataset structure on your machine, then report back to adjust the code:
```python
for p in list(KAGGLE_INPUT_DIR.rglob("*"))[:30]:
    print(p)
```
2. Verify the dataset name added in the section **+ Add Input** in the right corner of the notebook (correct slug `statsbomb-football-data`, because the mount path depends exactly on this slug).
3. If the dataset on Kaggle has file/folder names different from StatsBomb's original convention (e.g., renamed `events` to `event`), adjust the matching condition `"events"` / `"matches"` in the `discover_paths` function in cell 1.1 to match.

## Task 1 Results

File `/kaggle/working/shots_worldcup_2018_2022_raw.csv` contains all shot events of FIFA World Cup 2018 and 2022 (128 matches), already split into 3 field groups (identifiers / model features / validation fields). This is the input for **Task 2 — Thống kê mô tả** and **Task 3 — EDA** in the next step.


# Task 2 & 3 — Descriptive Statistics, Visualization & Exploratory Data Analysis (EDA)

**Input:** `shots_worldcup_2018_2022_raw.csv` — Task 1 output (3200 shots from 128 matches of World Cup 2018 + 2022).

**Notebook structure:**
- **Task 2** — Basic visualization (histogram, bar chart) + descriptive statistics (dispersion, quartile, z-score, box plot, skewness, kurtosis).
- **Task 3** — In-depth EDA (spatial heatmap, correlation heatmap, time trend with Moving Average, Control Chart, Correlation vs Regression, Correlation vs Causation, Random Sampling & Central Limit Theorem) — concluding with a design decision table for Task 4 (Preprocessing).

Note: `keeper_x`/`keeper_y` is missing in some shots (goalkeeper does not appear in `freeze_frame`) — known from Task 1, no need to repeat the check here. `outcome` and `statsbomb_xg` are only used for **interpretation/validation**, not included in any feature selection step.


## 0 — Setup

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from scipy import stats

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.titleweight"] = "bold"

CSV_PATH = "/kaggle/input/datasets/tunlcvcng/wc-2018-2022/shots_worldcup_2018_2022_raw.csv"  # change this if you are running elsewhere
df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
df.shape


## 2.1 — Data Overview

In [ ]:
print("Shape:", df.shape)
df.head(3)


In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)

fig, ax = plt.subplots(figsize=(6, 3))
missing.plot(kind="barh", ax=ax, color="#c0392b")
ax.set_title("Number of missing values by columns")
ax.set_xlabel("Number of missing rows")
plt.tight_layout()
plt.show()


## 2.2 — Bar chart: categorical variables distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

df["outcome"].value_counts().plot(kind="bar", ax=axes[0, 0], color="#2980b9")
axes[0, 0].set_title("Distribution of Outcome")
axes[0, 0].tick_params(axis="x", rotation=45)

df["body_part"].value_counts().plot(kind="bar", ax=axes[0, 1], color="#27ae60")
axes[0, 1].set_title("Distribution of Body Part")
axes[0, 1].tick_params(axis="x", rotation=45)

df["shot_type"].value_counts().plot(kind="bar", ax=axes[1, 0], color="#8e44ad")
axes[1, 0].set_title("Distribution of Shot Type")
axes[1, 0].tick_params(axis="x", rotation=45)

df["play_pattern"].value_counts().plot(kind="bar", ax=axes[1, 1], color="#d35400")
axes[1, 1].set_title("Distribution of Play Pattern")
axes[1, 1].tick_params(axis="x", rotation=60)

plt.tight_layout()
plt.show()


In [ ]:
top_teams = df["team_name"].value_counts().head(10)

fig, ax = plt.subplots(figsize=(8, 5))
top_teams.sort_values().plot(kind="barh", ax=ax, color="#16a085")
ax.set_title("Top 10 teams with the most shots (World Cup 2018 + 2022)")
ax.set_xlabel("Number of shots")
plt.tight_layout()
plt.show()


In [ ]:
season_labels = {3: "World Cup 2018", 106: "World Cup 2022"}
season_counts = df["season_id"].map(season_labels).value_counts()

fig, ax = plt.subplots(figsize=(5, 4))
season_counts.plot(kind="bar", ax=ax, color=["#e74c3c", "#3498db"])
ax.set_title("Number of shots by season")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

print(season_counts)


## 2.3 — Histogram: quantitative variables distribution

In [ ]:
numeric_cols = ["distance_to_goal", "angle_to_goal", "statsbomb_xg", "minute"]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.ravel()
colors = ["#2980b9", "#27ae60", "#c0392b", "#8e44ad"]

for ax, col, color in zip(axes, numeric_cols, colors):
    ax.hist(df[col].dropna(), bins=40, color=color, alpha=0.8)
    ax.set_title(f"Histogram: {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()


## 2.4 — Summary descriptive statistics table

In [ ]:
desc = df[numeric_cols].describe().T
desc


## 2.5 — Measuring dispersion (Dispersion): Std, Variance, Range, IQR

- **Standard Deviation / Variance**: measures how much the data fluctuates around the mean.
- **Range**: distance between the maximum and minimum values — sensitive to outliers.
- **IQR (Interquartile Range) = Q3 − Q1**: measures the dispersion of the "middle" of the data, less dominated by outliers than Range.

In [ ]:
dispersion_rows = []
for col in numeric_cols:
    s = df[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    dispersion_rows.append({
        "feature": col,
        "std": s.std(),
        "variance": s.var(),
        "range": s.max() - s.min(),
        "Q1": q1,
        "Q3": q3,
        "IQR": q3 - q1,
    })

dispersion_df = pd.DataFrame(dispersion_rows).set_index("feature")
dispersion_df


## 2.6 — Box-and-Whisker Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(data=df, x="outcome", y="distance_to_goal", ax=axes[0], palette="Set2")
axes[0].set_title("Distance to goal by Outcome")
axes[0].tick_params(axis="x", rotation=45)

sns.boxplot(data=df, x="body_part", y="statsbomb_xg", ax=axes[1], palette="Set3")
axes[1].set_title("xG by Body Part")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


## 2.7 — Measuring position: Quartile, Percentile & Z-score

Besides the calculated Q1/Q3, looking at other percentiles (e.g. 95th, 99th) tells us where the "extremely good chances" threshold lies. Then we use **Z-score** to flag unusual shots in terms of xG compared to the rest.

In [ ]:
percentiles = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
pct_table = df["statsbomb_xg"].quantile(percentiles)
pct_table


In [ ]:
df["xg_zscore"] = stats.zscore(df["statsbomb_xg"])

n_outliers = (df["xg_zscore"].abs() > 3).sum()
pct_outliers = n_outliers / len(df) * 100
print(f"Number of shots with |z-score| > 3: {n_outliers} ({pct_outliers:.1f}% total shots)")

df.sort_values("xg_zscore", ascending=False)[
    ["player_name", "team_name", "distance_to_goal", "statsbomb_xg", "outcome", "xg_zscore"]
].head(5)


## 2.8 — Skewness & Kurtosis

- **Skewness**: measures asymmetry. Positive = long right tail, Negative = long left tail, ≈0 = symmetric.
- **Kurtosis** (excess, compared to normal distribution = 0): positive = thicker/more peaked tails than normal (leptokurtic), negative = thinner/flatter tails (platykurtic).

In [ ]:
skew_kurt_rows = []
for col in ["distance_to_goal", "angle_to_goal", "statsbomb_xg"]:
    s = df[col].dropna()
    skew_kurt_rows.append({
        "feature": col,
        "skewness": stats.skew(s),
        "kurtosis_excess": stats.kurtosis(s),
    })

skew_kurt_df = pd.DataFrame(skew_kurt_rows).set_index("feature")
skew_kurt_df


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["distance_to_goal", "angle_to_goal", "statsbomb_xg"]):
    sns.histplot(df[col].dropna(), kde=True, ax=ax, color="#2c3e50")
    sk = skew_kurt_df.loc[col, "skewness"]
    ku = skew_kurt_df.loc[col, "kurtosis_excess"]
    ax.set_title(f"{col}\nskew={sk:.2f}, kurtosis={ku:.2f}")

plt.tight_layout()
plt.show()


## 3.1 — Check coordinate validity

In [ ]:
invalid_x = ~df["location_x"].between(0, 120)
invalid_y = ~df["location_y"].between(0, 80)
print("Number of x points outside [0,120]:", invalid_x.sum())
print("Number of y points outside [0,80]:", invalid_y.sum())


## 3.2 — Spatial heat map: shot position density on the pitch

Draw a simple pitch map (StatsBomb coordinates 120×80) as the background, then overlay a 2D density heatmap of all shot locations.

In [ ]:
def draw_pitch(ax):
    # Pitch boundaries
    ax.plot([0, 0, 120, 120, 0], [0, 80, 80, 0, 0], color="black", linewidth=1)
    # Halfway line
    ax.plot([60, 60], [0, 80], color="black", linewidth=1)
    # Right penalty area (where shots concentrate, StatsBomb always normalizes attacking direction to x=120)
    ax.plot([102, 102, 120], [18, 62, 62], color="black", linewidth=1)
    ax.plot([102, 120], [18, 18], color="black", linewidth=1)
    # 6-yard box
    ax.plot([114, 114, 120], [30, 50, 50], color="black", linewidth=1)
    ax.plot([114, 120], [30, 30], color="black", linewidth=1)
    ax.set_xlim(-2, 122)
    ax.set_ylim(-2, 82)
    ax.set_aspect("equal")
    ax.axis("off")


fig, ax = plt.subplots(figsize=(10, 7))
draw_pitch(ax)
sns.kdeplot(
    data=df, x="location_x", y="location_y",
    fill=True, cmap="Reds", alpha=0.75, levels=100, thresh=0.02, ax=ax,
)
ax.set_title("Shot location density — World Cup 2018 + 2022 (n=3200)")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
draw_pitch(ax)
ax.scatter(df["location_x"], df["location_y"], s=8, alpha=0.25, color="#c0392b")
ax.set_title("Scatter plot of all shot locations")
plt.tight_layout()
plt.show()


## 3.3 — Correlation Heatmap

In [ ]:
corr_cols = [
    "distance_to_goal", "angle_to_goal", "minute",
    "n_teammates_in_frame", "n_opponents_in_frame", "statsbomb_xg",
]
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation matrix between numeric variables")
plt.tight_layout()
plt.show()


## 3.4 — Line Chart & Moving Average (SMA, EMA): chance quality trend by match time

Group all shots by minute (0-90+), calculate average xG per minute. The raw series will be quite noisy (each minute only has a few dozen shots) so we apply **SMA (Simple Moving Average)** and **EMA (Exponential Moving Average)** to smooth and compare.

In [ ]:
by_minute = df.groupby("minute")["statsbomb_xg"].agg(["mean", "count"]).rename(columns={"mean": "avg_xg"})
by_minute = by_minute[by_minute.index <= 95]  # drop a few extremely rare stoppage time minutes to make the series cleaner

WINDOW = 5  # window size for SMA — tradeoff: smaller window tracks data closer but is noisier, larger window is smoother but lags more
by_minute["SMA"] = by_minute["avg_xg"].rolling(window=WINDOW, min_periods=1, center=True).mean()
by_minute["EMA"] = by_minute["avg_xg"].ewm(span=WINDOW, adjust=False).mean()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(by_minute.index, by_minute["avg_xg"], color="lightgray", label="Average xG by minute (raw)")
ax.plot(by_minute.index, by_minute["SMA"], color="#2980b9", linewidth=2, label=f"SMA (window={WINDOW})")
ax.plot(by_minute.index, by_minute["EMA"], color="#c0392b", linewidth=2, linestyle="--", label=f"EMA (span={WINDOW})")
ax.set_xlabel("Match Minute")
ax.set_ylabel("Average xG")
ax.set_title("Trend of chance quality (average xG) across match time")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(by_minute.index, by_minute["count"], color="#7f8c8d")
ax.set_xlabel("Match Minute")
ax.set_ylabel("Number of shots")
ax.set_title("Number of shots by match minute")
plt.tight_layout()
plt.show()


## 3.5 — Control Chart: detect matches with unusual chance quality

Consider each match as 1 "process output", calculate average xG per match, then build a control chart with the centerline (mean) and control limits UCL/LCL = mean ± 3×std across all 128 matches.

In [ ]:
match_avg_xg = df.groupby("match_id")["statsbomb_xg"].mean().reset_index()
match_avg_xg = match_avg_xg.merge(
    df[["match_id", "season_id"]].drop_duplicates(), on="match_id"
).sort_values("match_id").reset_index(drop=True)

mu = match_avg_xg["statsbomb_xg"].mean()
sigma = match_avg_xg["statsbomb_xg"].std()
ucl = mu + 3 * sigma
lcl = max(0, mu - 3 * sigma)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(match_avg_xg.index, match_avg_xg["statsbomb_xg"], marker="o", linestyle="-", color="#34495e", markersize=4)
ax.axhline(mu, color="green", linestyle="-", label=f"Mean = {mu:.3f}")
ax.axhline(ucl, color="red", linestyle="--", label=f"UCL (+3σ) = {ucl:.3f}")
ax.axhline(lcl, color="red", linestyle="--", label=f"LCL (-3σ) = {lcl:.3f}")

out_of_control = match_avg_xg[(match_avg_xg["statsbomb_xg"] > ucl) | (match_avg_xg["statsbomb_xg"] < lcl)]
ax.scatter(out_of_control.index, out_of_control["statsbomb_xg"], color="red", s=80, zorder=5, label="Out of control")

ax.set_xlabel("Match (by match_id order)")
ax.set_ylabel("Average xG / trận")
ax.set_title("Control Chart — Average xG per match (World Cup 2018 + 2022)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Number of out-of-control matches: {len(out_of_control)}")
match_avg_xg.loc[out_of_control.index]


## 3.6 — Correlation vs Regression

Correlation only measures the **strength/direction** of a linear relationship, it doesn't provide a prediction formula. Regression goes further: it estimates the best line equation to **predict** one variable from another.

In [ ]:
x = df["distance_to_goal"].values
y = df["statsbomb_xg"].values

slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(x, y, alpha=0.15, color="#2980b9", s=15)
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, intercept + slope * x_line, color="red", linewidth=2,
        label=f"y = {intercept:.3f} + ({slope:.4f})·x")
ax.set_xlabel("distance_to_goal")
ax.set_ylabel("statsbomb_xg")
ax.set_title(f"Correlation r = {r_value:.3f}  |  R² = {r_value**2:.3f}  |  p-value = {p_value:.2e}")
ax.legend()
plt.tight_layout()
plt.show()


## 3.7 — Correlation vs Causation

`under_pressure` and `statsbomb_xg` have a clear negative correlation — but does **defensive pressure directly reduce chance quality**, or is this just a correlation confounded by a third variable (confounder), for example: shots under pressure are often also shots from further away/tighter angles?

A simple test: compare `under_pressure` vs `statsbomb_xg` **within separate distance bins** (controlling for `distance_to_goal`) — if the difference remains clear after controlling for distance, it's supporting evidence for a true effect of pressure and not just confounding.

In [ ]:
raw_compare = df.groupby("under_pressure")["statsbomb_xg"].mean()
print("Raw comparison (not controlling for anything):")
print(raw_compare)
print()

df["dist_bin"] = pd.cut(df["distance_to_goal"], bins=[0, 10, 20, 30, 100],
                         labels=["0-10m", "10-20m", "20-30m", "30m+"])

controlled_compare = df.groupby(["dist_bin", "under_pressure"], observed=True)["statsbomb_xg"].mean().unstack()
print("Comparison after controlling for distance (dist_bin):")
controlled_compare


In [ ]:
controlled_compare.plot(kind="bar", figsize=(9, 5), color=["#2980b9", "#c0392b"])
plt.title("Average xG: Under Pressure vs Not, by distance bins")
plt.ylabel("Average xG")
plt.xticks(rotation=0)
plt.legend(title="under_pressure")
plt.tight_layout()
plt.show()


## 3.8 — Random Sample, Sample Mean & Central Limit Theorem

`statsbomb_xg` (the population here) is strongly right-skewed (skew ≈ 2.7, as seen in section 2.8) — not at all like a normal distribution. The Central Limit Theorem (CLT) states: even if the population itself is not normal, the **distribution of the sample mean** will approach a normal distribution as the sample size `n` becomes large enough. We illustrate this by taking repeated random samples with different sample sizes.

In [ ]:
population = df["statsbomb_xg"].dropna().values
pop_mean, pop_std = population.mean(), population.std()
print(f"Population: mean={pop_mean:.4f}, std={pop_std:.4f}, skew={stats.skew(population):.4f}")

rng = np.random.default_rng(42)
N_REPEATS = 3000
sample_sizes = [5, 30, 100]

fig, axes = plt.subplots(1, len(sample_sizes), figsize=(15, 4.5), sharey=False)

for ax, n in zip(axes, sample_sizes):
    sample_means = np.array([
        rng.choice(population, size=n, replace=True).mean() for _ in range(N_REPEATS)
    ])
    empirical_se = sample_means.std()
    byretical_se = pop_std / np.sqrt(n)

    sns.histplot(sample_means, kde=True, ax=ax, color="#2980b9")
    ax.set_title(f"n = {n}\nempirical SE={empirical_se:.4f} | theoretical SE={byretical_se:.4f}\nskew sample mean={stats.skew(sample_means):.2f}")
    ax.axvline(pop_mean, color="red", linestyle="--")

plt.tight_layout()
plt.show()


## 3.9 — Summary: design decision table for Task 4

In [ ]:
summary = pd.DataFrame([
    {
        "Finding from Task 2/3": "distance_to_goal, angle_to_goal, statsbomb_xg have very different scales (meters vs radians vs 0-1 probability)",
        "Decision for Task 4": "Need to experiment scaled vs unscaled before K-Means (per initial design)",
    },
    {
        "Finding from Task 2/3": "statsbomb_xg is strongly right-skewed (skew≈2.7, kurtosis≈7.1); Z-score>3 flags ~5% of data instead of ~0.3%",
        "Decision for Task 4": "Do not solely rely on Z-score to remove outliers on heavily skewed variables; statsbomb_xg is still only used to validate, not fed into X",
    },
    {
        "Finding from Task 2/3": "No candidate feature pair is > 0.8 correlated in the corr heatmap",
        "Decision for Task 4": "No need to drop features due to information overlap yet",
    },
    {
        "Finding from Task 2/3": "keeper_x/keeper_y missing in 125/3200 shots (~3.9%)",
        "Decision for Task 4": "Can be kept (small missing ratio) — consider dropping missing rows or imputing if feeding keeper location into X",
    },
    {
        "Finding from Task 2/3": "Sample size 3200 shots, large enough for Elbow/Silhouette/Gap Statistic + k-fold CV per CLT (n=100 is almost normal)",
        "Decision for Task 4": "No need to expand data further before entering modeling",
    },
    {
        "Finding from Task 2/3": "under_pressure is associated with lower xG, holds after controlling for distance (except the 20-30m bin)",
        "Decision for Task 4": "Keep under_pressure as a feature for X (contains true signal, not just confounding from distance)",
    },
])

pd.set_option("display.max_colwidth", 100)
summary


## Task 2 & 3 Results

All the plots, summary statistics, and the design decision table in section 3.9 are direct inputs for **Task 4 — Preprocessing** (data scaling, encoding categoricals, handling missing values, creating `X_scaled`/`X_unscaled`) agreed upon in the overall plan previously.
